# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** *xx*  
**Kaggle challenge:** *xx* (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "*xx*"  

**Author 1 (SCIPER):** *Student Name 1 (xxxxx)*  
**Author 2 (SCIPER):** *Student Name 2 (xxxxx)*  
**Author 3 (SCIPER):** *Student Name 3 (xxxxx)*  

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

# Chocolate Detection and Counting
This report focuses on the challenge of detecting and counting chocolates in images. The pipeline uses a Faster R-CNN model with a custom backbone and head, trained on a COCO-style dataset.

## 1. Dataset Preparation
The dataset is in COCO format, with annotations for 13 chocolate classes and 1 background class. The `ChocolateCocoDataset` class loads the dataset and applies transformations.

In [ ]:
# Load the Dataset
from src.dataset import ChocolateCocoDataset
from src.transforms import get_transform

dataset = ChocolateCocoDataset(
    img_folder="dataset_project_iapr2025/images",
    coco_json="dataset_project_iapr2025/train_coco_dataset.json",
    transforms=get_transform(train=True)
)

print(f"Number of images: {len(dataset)}")
print(f"Classes: {dataset.labels}")

## 2. Model Definition
The model is a Faster R-CNN with a ResNet-18 backbone, Feature Pyramid Network (FPN), and a lightweight TwoMLPHead. It predicts bounding boxes and class probabilities for chocolates.

In [ ]:
# Load the Model
from src.model import get_model

num_classes = 14  # 13 chocolates + 1 background
model = get_model(num_classes)
print(model)

### Explanation of Model Components
1. **Backbone (ResNet-18)**:
   - Extracts features from the input image.
   - Layers 2, 3, and 4 are used because they capture low-level (edges), mid-level (shapes), and high-level (object parts) features, respectively.

2. **Feature Pyramid Network (FPN)**:
   - Combines features from different scales into a unified representation.
   - Reduces all feature maps to 64 channels for computational efficiency.

3. **Region Proposal Network (RPN)**:
   - Proposes candidate regions (bounding boxes) where objects might be located.
   - Uses anchors of different sizes and aspect ratios to handle chocolates of varying shapes and sizes.

4. **RoI Pooling**:
   - Extracts fixed-size feature maps for each proposed region using multi-scale pooling.

5. **TwoMLPHead**:
   - A lightweight two-layer fully connected head for classification and bounding box regression.

6. **Final Predictor**:
   - Predicts class probabilities and bounding box adjustments.

## 3. Training
The model is trained using cross-validation and final retraining. The loss function combines classification, bounding box regression, and RPN losses.

In [ ]:
# Train the Model
from src.train import run_training
import argparse

args = argparse.Namespace(
    train_imgs="dataset_project_iapr2025/images",
    coco_json="dataset_project_iapr2025/train_coco_dataset.json",
    output_root="output",
    run_name="ultimate_run",
    batch_size=16,
    num_workers=4,
    epochs=10,
    lr=0.005,
    step_size=5,
    gamma=0.1,
    patience=3,
    seed=42,
    k_folds=5,
    final_epochs=10,
    skip_cv=False,
    conf_thresh_score=0.5,
    conf_thresh_vis=0.5,
    visual_samples=4
)

run_training(args)

## 4. Inference
The trained model is used to detect and count chocolates in test images. Predictions are filtered based on confidence threshold, and results are saved to a CSV file.

In [ ]:
# Run Inference
from src.infer import run_inference

args = argparse.Namespace(
    test_folder="dataset_project_iapr2025/images",
    model_path="output/ultimate_run_final/best_full.pth",
    output_csv="submission_ultimate.csv",
    label_names="Amandina,Arabia,Comtesse,Creme_brulee,Jelly_Black,Jelly_Milk,Jelly_White,Noblesse,Noir_authentique,Passion_au_lait,Stracciatella,Tentation_noir,Triangolo",
    conf_thresh=0.5
)

run_inference(args)

## 5. Visualization
Visualizations are used to analyze the model's performance. Ground truth and predicted bounding boxes are drawn on images.

In [ ]:
# Visualize Predictions
from src.utils import visualize_predictions

visualize_predictions(
    model=model,
    dataset=dataset,
    class_names=dataset.labels,
    device="cuda" if torch.cuda.is_available() else "cpu",
    epoch=1,
    out_dir="output/visuals",
    n_samples=4,
    conf_thresh=0.5
)

## 6. Quantitative Analysis
The model's performance is evaluated using metrics like F1-score, IoU, and confusion matrix.

In [ ]:
# Compute Metrics
from src.utils import compute_counts_from_preds, image_f1
import numpy as np

# Example: Compute F1-score for a batch of predictions
true_counts = [1, 2, 0, 0, 1]  # Example ground truth counts
pred_counts = [1, 1, 0, 0, 2]  # Example predicted counts

f1 = image_f1(np.array(true_counts), np.array(pred_counts))
print(f"F1-score: {f1:.4f}")

## 7. Conclusion
The Faster R-CNN model successfully detects and counts chocolates in images. The pipeline demonstrates strong performance, with high F1-scores and accurate visualizations.